# QA Metrics — Polished Workflow

This notebook presents a concise, production-ready workflow with minimal commentary and clear sectioning.


## Overview
- Organized into logical steps with succinct headers.
- Outputs hidden by default to reduce clutter.
- Each code block has a short descriptive comment.



# Q/A Metrics: BERTScore, ROUGE-L, CIDEr-R

This notebook reads a JSON file of question–answer pairs and computes three text-matching metrics **per row** comparing the model's **Answer** to the **Gold Answer**:

- **BERTScore** (P/R/F1)
- **ROUGE-L** (F1)
- **CIDEr-R** (CIDEr-D variant from COCO caption eval; reported as a single score)

> **Inputs expected in each JSON object:** `"Question"`, `"Answer"`, and `"Gold Answer"` (case-sensitive, space included).

## How to use
1. Run the first cell to install dependencies (internet required).
2. Set `JSON_PATH` to your file if it differs.
3. Run all cells. You'll get a DataFrame with all scores and a CSV export.


### Computation

In [ ]:

# === Install dependencies (run once) ===
# If you're in an environment without internet access, install these packages beforehand.
%pip install --quiet bert-score rouge-score git+https://github.com/tylin/coco-caption#subdirectory=pycocoevalcap


### Setup & imports

In [ ]:

# === Imports ===
import json
from pathlib import Path
import pandas as pd
from tqdm import tqdm

from bert_score import score as bert_score
from rouge_score import rouge_scorer
from pycocoevalcap.cider.cider import Cider

# Path to your JSON file
JSON_PATH = Path(r"./Gemma-3-4b-MLLM-only-answers-custom-retrieval-openai-with-system-prompt.json")
assert JSON_PATH.exists(), f"File not found: {JSON_PATH}"

# Read JSON
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

# Normalize into DataFrame
df = pd.json_normalize(data)

# Ensure required columns exist (exact names per your file)
required_cols = ["Question", "Answer", "Gold Answer"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required column(s): {missing}. Present columns: {list(df.columns)}")

# Coerce to strings (handles NAs, numbers, lists, etc.)
df["Answer"] = df["Answer"].astype(str)
df["Gold Answer"] = df["Gold Answer"].astype(str)

df.head(3)


## BERTScore (precision, recall, F1)

### Modeling & evaluation

In [ ]:

# BERTScore for each pair (Answer vs Gold Answer)
# You can change model_type (e.g., 'roberta-large', 'microsoft/deberta-xlarge-mnli') for potentially better results.
cands = df["Answer"].tolist()
refs  = df["Gold Answer"].tolist()

P, R, F1 = bert_score(cands, refs, model_type="roberta-large", lang="en", rescale_with_baseline=True)
df["BERTScore_P"]  = P.tolist()
df["BERTScore_R"]  = R.tolist()
df["BERTScore_F1"] = F1.tolist()

df[["BERTScore_P","BERTScore_R","BERTScore_F1"]].head(3)


## ROUGE-L (F1)

### Exploratory checks

In [ ]:

# ROUGE-L using rouge_score
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

rouge_l_scores = []
for cand, ref in zip(df["Answer"], df["Gold Answer"]):
    s = rouge.score(ref, cand)  # order: target first, prediction second
    rouge_l_scores.append(s["rougeL"].fmeasure)

df["ROUGE_L_F1"] = rouge_l_scores
df["ROUGE_L_F1"].head(3)


## CIDEr-R (CIDEr-D)

### Exploratory checks

In [ ]:

# CIDEr from pycocoevalcap
# It expects dicts: {id: [list of reference strings]} for gts, and {id: [candidate string]} for res.
cider_scorer = Cider()

cider_scores = []
for i, (cand, ref) in enumerate(zip(df["Answer"], df["Gold Answer"])):
    gts = {i: [ref]}
    res = {i: [cand]}
    score, _ = cider_scorer.compute_score(gts, res)
    # compute_score returns (mean_score, per_sample_scores); when single item, mean == that score
    cider_scores.append(float(score))

df["CIDEr_R"] = cider_scores
df["CIDEr_R"].head(3)


## Final DataFrame and Export

### Exploratory checks

In [ ]:

# Select and order useful columns
score_cols = ["BERTScore_P","BERTScore_R","BERTScore_F1","ROUGE_L_F1","CIDEr_R"]
out_cols = ["Question","Answer","Gold Answer"] + score_cols
out_df = df[out_cols].copy()

# Save to CSV
csv_path = JSON_PATH.with_suffix(".scored.csv")
out_df.to_csv(csv_path, index=False, encoding="utf-8")

print(f"Saved scores to: {csv_path}")
out_df.head(10)
